In [55]:
import pandas as pd
import numpy as np

In [32]:
df = pd.read_csv("../Data/cleaned_app.csv")

In [33]:
df

,user_rating,review,parent_asin,user_id,verified_purchase
0,5,Work great,B01N0TQ0OH,AGKHLEW2SOWHNMFQIJGBECAF7INQ,True
1,5,excellent product,B07DD37QPZ,AHWWLSPCJMALVHDDVSUGICL6RUCA,True
2,5,Happy customer!,B082W3Z9YK,AHZIJGKEWRTAEOZ673G5B3SNXEGQ,True
3,5,Amazing value,B078W2BJY8,AFGUPTDFAWOHHL4LZDV27ERDNOYQ,True
4,5,Dryer parts,B08C9LPCQV,AELFJFAXQERUSMTXJQ6SYFFRDWMA,True
...,...,...,...,...,...
2128600,5,Accurate description,B097948QRP,AG6IN4MOTWF3743PKIPHYA2S7GXA,True
2128601,3,Not compatible with Nespresso U Machine,B0C6XG2JSG,AHVKX5FONDMQVOA7XLMPAH6EGZ2Q,True
2128602,5,Works with Sears Kenmore model 36275585891,B07QKBMPG2,AEYETSNK5VL6ZSLN32EE6VCOAYFA,True
2128603,5,Perfect little ice maker!,B07H7SGQ52,AHIJLNIXWVFQFWJV3OGGQOHONGMQ,True


In [47]:
# Clean
ratings = df.copy()
ratings = ratings.drop_duplicates(subset=["user_id", "parent_asin"])
ratings["user_rating"] = pd.to_numeric(ratings["user_rating"], errors="coerce")
ratings = ratings.dropna(subset=["user_rating"])

In [48]:
ratings["user_id"] = ratings["user_id"].astype("category")
ratings["parent_asin"] = ratings["parent_asin"].astype("category")
ratings["user_rating"] = ratings["user_rating"].astype(float)

user_codes = ratings["user_id"].cat.codes
item_codes = ratings["parent_asin"].cat.codes

In [49]:
num_users = len(ratings["user_id"].cat.categories)
num_items = len(ratings["parent_asin"].cat.categories)

print(f"Users: {num_users}, Items: {num_items}")

Users: 1755732, Items: 94319


In [51]:
from scipy.sparse import csr_matrix
R = csr_matrix(
    (ratings["user_rating"], (user_codes, item_codes)),
    shape=(num_users, num_items),
    dtype=float
)


In [52]:
from scipy.sparse.linalg import svds
# =========================
# 4. APPLY SVD
# =========================
k = 20  # latent features

# Ensure k is valid
k = min(k, min(R.shape) - 1)

U, sigma, Vt = svds(R, k=k)

# Convert sigma to diagonal
sigma = np.diag(sigma)

# Compute item embeddings
V = Vt.T   # (items × k)

In [53]:
# =========================
# 5. CREATE MAPPINGS
# =========================
user_map = dict(enumerate(ratings["user_id"].cat.categories))
item_map = dict(enumerate(ratings["parent_asin"].cat.categories))

user_inv_map = {v: k for k, v in user_map.items()}
item_inv_map = {v: k for k, v in item_map.items()}



In [64]:
products_df = pd.read_csv("../Data/cleaned_meta_app.csv")

asin_to_name = dict(
    zip(products_df["Parent_ASIN"], products_df["Product_Name"])
)

In [65]:
# =========================
# 6. RECOMMENDATION FUNCTION
# =========================
def recommend_svd(user_id, top_n=5):
    
    if user_id not in user_inv_map:
        print("User not found")
        return None
    
    user_idx = user_inv_map[user_id]
    
    # Get user embedding
    user_vec = U[user_idx]   # (k,)
    
    # Compute scores for all items
    scores = V @ user_vec    # (num_items,)
    
    # Remove already interacted items
    user_interactions = R[user_idx].toarray().flatten()
    
    scores[user_interactions > 0] = -np.inf
    
    # Get top N items
    top_items = np.argpartition(scores, -top_n)[-top_n:]
    top_items = top_items[np.argsort(-scores[top_items])]
    
    # Convert index → ASIN → Name
    recommendations = []
    
    for i in top_items:
        asin = item_map[i]
        name = asin_to_name.get(asin, "Unknown Product")
        recommendations.append(name)
    

    return recommendations

In [66]:
sample_user = ratings["user_id"].iloc[0]
    
recs = recommend_svd(sample_user, top_n=5)